# Q-Mini Add/Sub v1 — From-Scratch English Training

This notebook builds the **new independent model line** inside the Q-mini repository. It does not load or modify the old Q-Mini Light model, its checkpoints, tokenizer or datasets.

Important: training is **OFF by default**. Review source licenses first, explicitly enable pretraining, then enable SFT only after checking the pretraining metrics. CPU is the safe default; choose CUDA only when you intentionally want to use a GPU.

Data sources: pinned WikiText-103 raw for language pretraining, reviewed/non-synthetic OASST1 conversations and human-authored Dolly instruction pairs. TinyStories and model-generated synthetic corpora are excluded. DailyDialog is excluded because its license has non-commercial/share-alike restrictions.

The 5-symbol weight alphabet needs 3-bit storage. The training graph uses floating-point optimizer arithmetic; the complete multiply-free inference runtime is still under development. Do not use the training graph as a deployed inference engine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lion36508/Q-mini-light-0.7M.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/Q-mini-light-0.7M')
if not (REPO_DIR / 'qmini_addsub').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/QMini-Light-AddSub-v1')
DATA_DIR = DRIVE_ROOT / 'data'
CACHE_DIR = DRIVE_ROOT / 'cache'
ARTIFACT_DIR = DRIVE_ROOT / 'artifacts'
PRETRAIN_DIR = DRIVE_ROOT / 'runs' / 'pretrain'
SFT_DIR = DRIVE_ROOT / 'runs' / 'sft'
for folder in (DATA_DIR, CACHE_DIR, ARTIFACT_DIR, PRETRAIN_DIR, SFT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Install data/tokenizer tools without replacing Colab's preinstalled PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24', 'tokenizers>=0.15', 'huggingface_hub>=0.28', 'pyarrow>=15.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', '-e', str(REPO_DIR)], check=True)

import torch
print('PyTorch:', torch.__version__)
print('Available CUDA:', torch.cuda.is_available())
print('New model Drive root:', DRIVE_ROOT)


## Source-license approval

Before downloading, review the dataset cards and obligations: WikiText-103 (CC BY-SA / GFDL metadata), OASST1 (Apache-2.0), Dolly 15k (CC BY-SA 3.0). Share-alike and attribution obligations may affect redistribution. This notebook makes no legal determination.

In [ ]:
ACCEPT_SOURCE_LICENSES = False  # Change to True only after reviewing the terms above.
if not ACCEPT_SOURCE_LICENSES:
    raise RuntimeError('Review the three source licenses, then set ACCEPT_SOURCE_LICENSES=True.')

prepare_command = [
    sys.executable, 'scripts/prepare_qmini_addsub_data.py',
    '--output', str(DATA_DIR),
    '--cache', str(CACHE_DIR),
    '--accept-source-licenses',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
print('Verified English corpus prepared. No model training has started.')


In [ ]:
TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.json'
subprocess.run([
    sys.executable, 'scripts/train_qmini_addsub_tokenizer.py',
    '--data-dir', str(DATA_DIR),
    '--output', str(TOKENIZER_PATH),
    '--vocab-size', '512',
], cwd=REPO_DIR, check=True)

subprocess.run([
    sys.executable, 'scripts/encode_qmini_addsub_streams.py',
    '--data-dir', str(DATA_DIR),
    '--tokenizer', str(TOKENIZER_PATH),
], cwd=REPO_DIR, check=True)
print('Tokenizer and verified train/validation/test token streams are ready.')


## Stage 1 — base English pretraining

Start from random initialization. This stage writes checkpoints only under the separate Add/Sub v1 Drive folder. The default is CPU and a small trial run. Raise the step count only after reviewing held-out loss and available compute.

In [ ]:
START_PRETRAIN = False  # Change to True only when ready to begin a new model training run.
DEVICE = 'cpu'          # Safe default. Set to 'cuda' only when you intentionally use a GPU.
PRETRAIN_STEPS = 1000
BATCH_SIZE = 2 if DEVICE == 'cpu' else 8
SEQUENCE_LENGTH = 256
PRETRAIN_RESUME = None  # Set to a checkpoint from this Add/Sub v1 run only.

if START_PRETRAIN:
    command = [
        sys.executable, '-m', 'qmini_addsub.train',
        '--train-bin', str(DATA_DIR / 'pretrain_train.u16'),
        '--valid-bin', str(DATA_DIR / 'pretrain_validation.u16'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output', str(PRETRAIN_DIR),
        '--device', DEVICE,
        '--steps', str(PRETRAIN_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--sequence-length', str(SEQUENCE_LENGTH),
    ]
    if PRETRAIN_RESUME:
        command += ['--resume', str(PRETRAIN_RESUME)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('Pretraining is OFF. Set START_PRETRAIN=True to start from random initialization.')


## Stage 2 — supervised instruction fine-tuning

Only run after Stage 1 has produced a non-zero-step Add/Sub v1 checkpoint and its validation loss has been inspected. The SFT trainer rejects legacy Q2-v2 checkpoints and random step-zero artifacts.

In [ ]:
START_SFT = False
SFT_STEPS = 500
SFT_DEVICE = DEVICE
PRETRAIN_CHECKPOINT = None  # Set to an Add/Sub v1 pretraining checkpoint path.
SFT_RESUME = None            # Optional Add/Sub v1 SFT checkpoint only.

if START_SFT:
    if PRETRAIN_CHECKPOINT is None:
        candidates = sorted(PRETRAIN_DIR.glob('checkpoint-step-*.pt'))
        if not candidates:
            raise FileNotFoundError('No Add/Sub v1 pretraining checkpoint found. Finish Stage 1 first.')
        PRETRAIN_CHECKPOINT = candidates[-1]
    command = [
        sys.executable, '-m', 'qmini_addsub.sft',
        '--base-checkpoint', str(PRETRAIN_CHECKPOINT),
        '--tokenizer', str(TOKENIZER_PATH),
        '--train-jsonl', str(DATA_DIR / 'sft_train.jsonl'),
        '--valid-jsonl', str(DATA_DIR / 'sft_validation.jsonl'),
        '--output', str(SFT_DIR),
        '--device', SFT_DEVICE,
        '--steps', str(SFT_STEPS),
    ]
    if SFT_RESUME:
        command += ['--resume', str(SFT_RESUME)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('SFT is OFF. Enable only after checking the new model pretraining checkpoint.')


In [ ]:
print('Add/Sub v1 data:', DATA_DIR)
print('Add/Sub v1 tokenizer:', TOKENIZER_PATH)
print('Pretraining checkpoints:', PRETRAIN_DIR)
print('SFT checkpoints:', SFT_DIR)
print('Legacy model and legacy Drive folder were not selected by this notebook.')
print('Full multiply-free model inference/export is not yet complete.')
